In [9]:
# | default_exp models

# Model Definition

> Defining a simple MLP model for binary classification

In [10]:
# | export
from torch import nn
from typing import Literal
from copy import deepcopy
import torch

In [11]:
# | export
class MLP(nn.Module):
    """
    Simple MLP module for binary classification.
    """

    def __init__(
        self,
        config: list = [8],  # Number of neurons per linear layer.
        in_features: int = 2,  # Number of input features to the model.
        out_features: int = 2,  # Number of output features.
        nonlinearity: Literal["tanh", "relu"] = "tanh",  # Nonlinearity to use.
    ):
        super().__init__()

        if nonlinearity == "tanh":
            act = nn.Tanh()
        elif nonlinearity == "relu":
            act = nn.ReLU()
        else:
            raise NotImplementedError()

        # Init container
        self.features = nn.Sequential()

        # First layer
        self.features.append(
            nn.Linear(in_features=in_features, out_features=config[0], bias=True)
        )
        self.features.append(deepcopy(act))

        for i, _ in enumerate(config[1:], start=1):
            self.features.append(
                nn.Linear(in_features=config[i - 1], out_features=config[i], bias=True)
            )
            self.features.append(deepcopy(act))

        # Last layer
        self.features.append(
            nn.Linear(in_features=config[-1], out_features=out_features, bias=True)
        )

    def forward(self, x):
        if x.ndim > 2:
            x = torch.flatten(x, start_dim=1)
        logits = self.features(x)
        return logits

In [12]:
model = MLP(config=[3, 2], in_features=2, nonlinearity="relu")
model

MLP(
  (features): Sequential(
    (0): Linear(in_features=2, out_features=3, bias=False)
    (1): ReLU()
    (2): Linear(in_features=3, out_features=2, bias=True)
    (3): ReLU()
    (4): Linear(in_features=2, out_features=2, bias=False)
  )
)

In [13]:
# |export
def init_weights(
    module: nn.Module,  # Module to apply this method on.
):
    "Initialize the weights of a model"
    if isinstance(module, nn.Linear):
        nn.init.xavier_normal_(module.weight)
        if module.bias is not None:
            module.bias.data.fill_(0.01)

In [14]:
_ = model.apply(init_weights)